## Goal
Build a canonical resume from an existing resume in JSON. Validate conversion process by converting back into a resume.
- Stage 1 was developing the Master Resume doc which covered entire career history.

In [1]:
%run ./init_notebook.py


Repo root: /Users/douglasdaly/GitHub/Generative-AI
Added src to sys.path: /Users/douglasdaly/GitHub/Generative-AI/src
Resume builder notebooks: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder
Artifacts: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder/artifacts


In [2]:
from pathlib import Path
import json
from openai import OpenAI
from dotenv import load_dotenv
from genai_demos.resume_builder.config import SOURCE_DIR, LAYOUT_DIR, CONTRACT_DIR, ARTIFACT_DIR
from genai_demos.resume_builder.helpers import (
    extract_docx_text,
    load_json,
    parse_json_response,
    save_json
)
from genai_demos.resume_builder.renderer import (
    load_yaml,
    render_resume,
    convert_docx_to_pdf,
)

ARTIFACT_DIR.mkdir(exist_ok=True)


master_docx_path = SOURCE_DIR / "Master Resume.docx"
schema_path = CONTRACT_DIR / "Canonical Resume Schema.md"
canonical_master_resume_path = ARTIFACT_DIR / "canonical_master_resume.json"

load_dotenv()
client = OpenAI()
MODEL = "gpt-4.1-mini"



In [3]:
master_resume_text = extract_docx_text(master_docx_path)
schema_text = schema_path.read_text(encoding="utf-8")

In [4]:
# Exclude career archive section for now
resume_end = master_resume_text.find("CAREER ARCHIVE")
master_resume_text = master_resume_text[:resume_end]

In [5]:
def build_canonical_resume_prompt(master_resume_text, schema_text):
    return f"""
Convert the provided master resume into canonical resume JSON.

Follow the canonical resume schema exactly.

Important rules:
- Preserve career evidence.
- Do not shorten accomplishments.
- Do not optimize for a target role.
- Do not add facts not present in the master resume.
- Put resume content into structured sections.
- Include a stable section_id for every section.
- Put non-resume background material into an Archive section if needed.
- Preserve CONTEXT: lines as role_context or summary_context, not as ordinary bullets.
- Use role_context for role-level CONTEXT: lines.
- Use summary_context for subsection-level CONTEXT: lines, if present.
- Do not render or rewrite CONTEXT: labels in the JSON value; store only the explanatory text.
- Return valid JSON only.

Canonical Resume Schema:
{schema_text}

Master Resume:
{master_resume_text}
"""

In [6]:
prompt = build_canonical_resume_prompt(master_resume_text, schema_text)

In [7]:
response = client.responses.create(
    model=MODEL,
    input=prompt,
    temperature=0,
    text={
        "format": {"type": "json_object"}
    },
)

canonical_resume = parse_json_response(response)

In [8]:
CANONICAL_SECTION_IDS = {
    "Summary": "SUMMARY",
    "Core Technologies": "TECHNOLOGIES",
    "Professional Experience": "EXPERIENCE",
    "Selected Projects": "PROJECTS",
    "Education": "EDUCATION",
    "Patents & Recognition": "RECOGNITION",
    "Certifications": "CERTIFICATIONS",
    "Career Archive": "CAREER_ARCHIVE",
    "Speaking & Teaching": "SPEAKING_TEACHING",
    "Awards & Recognition": "AWARDS",
}


def normalize_section_id(value: str) -> str:
    return str(value).strip().upper()


def add_canonical_section_ids(resume: dict) -> dict:
    """
    Add stable section_id values to canonical resume sections.

    This is deterministic and intentionally strict. If a section heading changes,
    update CANONICAL_SECTION_IDS rather than silently guessing.
    """
    for section in resume.get("sections", []):
        heading = section.get("heading")

        if heading not in CANONICAL_SECTION_IDS:
            raise ValueError(
                f"Unexpected canonical section heading: {heading!r}. "
                "Add it to CANONICAL_SECTION_IDS if it is valid."
            )

        section["section_id"] = CANONICAL_SECTION_IDS[heading]

    return resume


def validate_canonical_resume(resume: dict) -> bool:
    assert "header" in resume
    assert "sections" in resume
    assert isinstance(resume["sections"], list)

    seen_section_ids = set()

    for section in resume["sections"]:
        assert "section_id" in section
        assert "heading" in section
        assert "type" in section
        assert "content" in section

        section_id = normalize_section_id(section["section_id"])

        if section_id in seen_section_ids:
            raise ValueError(f"Duplicate section_id found: {section_id}")

        seen_section_ids.add(section_id)

    return True


canonical_resume = add_canonical_section_ids(canonical_resume)
validate_canonical_resume(canonical_resume)

[
    {
        "section_id": section.get("section_id"),
        "heading": section.get("heading"),
        "type": section.get("type"),
    }
    for section in canonical_resume.get("sections", [])
]

[{'section_id': 'SUMMARY', 'heading': 'Summary', 'type': 'paragraph'},
 {'section_id': 'TECHNOLOGIES',
  'heading': 'Core Technologies',
  'type': 'subsections'},
 {'section_id': 'EXPERIENCE',
  'heading': 'Professional Experience',
  'type': 'experience'},
 {'section_id': 'PROJECTS',
  'heading': 'Selected Projects',
  'type': 'experience'},
 {'section_id': 'EDUCATION', 'heading': 'Education', 'type': 'subsections'},
 {'section_id': 'RECOGNITION',
  'heading': 'Patents & Recognition',
  'type': 'bullet'}]

In [9]:
save_json(
    canonical_resume,
    ARTIFACT_DIR / "canonical_master_resume.json"
)

# Phase 2 Summary: Canonical Resume Schema
## Objective

Define a stable, renderer-independent representation of resume content that can serve as the foundation for all future resume generation activities.

## Artifacts Produced

### Contracts

* Canonical Resume Schema.md

### Data

* canonical_master_resume.json

## Key Design Decisions

### Separate Content from Presentation

Resume content is stored in canonical JSON.

Presentation concerns such as fonts, spacing, alignment, bolding, and page layout are defined in YAML layout files and renderer logic.

### Keep the Schema Small

The schema intentionally uses a small number of content types:

* paragraph
* bullet
* inline_list
* subsections
* experience

Specialized structures such as education, certifications, awards, patents, and technology categories are represented using existing schema elements rather than introducing new content types.

### Use Subsections as the Primary Reusable Structure

The subsection object became the primary mechanism for representing labeled child records.

Examples include:

* Core Technologies
* Core Expertise
* Education
* Patents
* Awards
* Certifications
* Client engagements within consulting roles

### Renderer Independence

The canonical resume contains semantic information only.

Renderers determine visual presentation.

The same canonical JSON can be rendered using multiple layouts without modifying the underlying content.

## Validation Results

The canonical resume was successfully rendered to both DOCX and PDF formats.

Schema issues discovered during rendering led to several improvements, including:

* replacing flattened education bullets with structured subsections
* simplifying content types
* strengthening subsection semantics
* removing presentation concerns from content artifacts

The successful round-trip from:

Master Resume
→ Canonical Resume JSON
→ DOCX/PDF

demonstrates that the schema is sufficiently expressive for resume generation.

## Lessons Learned

The most important lesson from this phase was the importance of defining contracts before generation.

The renderer exposed weaknesses in the schema much earlier than downstream resume-generation workflows would have.

By validating the schema through rendering, structural issues were discovered and resolved before building archetypes, capability extraction, scoring pipelines, and resume-generation workflows.

## Exit Criteria

Phase 2 is considered complete when:

* the schema is documented
* the canonical resume can be generated
* the canonical resume can be rendered successfully
* layout changes can be made without modifying resume content

These criteria have been satisfied.

## Next Phase

Phase 3 focuses on market discovery.

Inputs:

* canonical_master_resume.json
* job descriptions

Output:

* target_archetype.json

The objective is to identify the capabilities, themes, priorities, and evidence patterns most valued by the target market before evaluating resume content.
